# Cart2Insights — Step 7: Feature Engineering

This notebook creates meaningful business features from the cleaned
e-commerce data stored in the MySQL database.

In [1]:
import os
import pandas as pd
import numpy as np

from sqlalchemy import create_engine, text
from dotenv import load_dotenv

In [2]:
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

print("Database:", DB_NAME)

Database: cart2insights


In [3]:
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected database:", DB_NAME)

c:\Users\Zoya khan\AppData\Local\Programs\Python\Python37\lib\site-packages\pymysql\_auth.py:9: CryptographyDeprecationWarning: Python 3.7 is no longer supported by the Python core team and support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.7.
  from cryptography.hazmat.backends import default_backend


Connected database: cart2insights


Sql setup of feature table

In [68]:
from sqlalchemy import text

In [69]:
with engine.connect() as connection:
    result = connection.execute("SELECT 1")
    print("MySQL connection successful.")

MySQL connection successful.


In [70]:
order_features_sql = """
CREATE OR REPLACE VIEW order_features AS
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    ROUND(COALESCE(ov.item_price_total, 0), 2) AS item_price_total,
    ROUND(COALESCE(ov.freight_total, 0), 2) AS freight_total,
    ROUND(COALESCE(ov.order_total_value, 0), 2) AS order_total_value,

    CASE
        WHEN o.order_delivered_customer_date IS NOT NULL
        THEN TIMESTAMPDIFF(
            DAY,
            o.order_purchase_timestamp,
            o.order_delivered_customer_date
        )
        ELSE NULL
    END AS delivery_days,

    CASE
        WHEN o.order_delivered_customer_date IS NOT NULL
             AND o.order_estimated_delivery_date IS NOT NULL
        THEN DATEDIFF(
            o.order_delivered_customer_date,
            o.order_estimated_delivery_date
        )
        ELSE NULL
    END AS delivery_delay_days,

    CASE
        WHEN o.order_delivered_customer_date IS NULL
            THEN 'Not Delivered'

        WHEN o.order_estimated_delivery_date IS NULL
            THEN 'Delivered - No Estimate'

        WHEN DATEDIFF(
            o.order_delivered_customer_date,
            o.order_estimated_delivery_date
        ) > 0
            THEN 'Late'

        WHEN DATEDIFF(
            o.order_delivered_customer_date,
            o.order_estimated_delivery_date
        ) = 0
            THEN 'On Time'

        ELSE 'Early'
    END AS delivery_status,

    CASE
        WHEN o.order_delivered_customer_date IS NULL
             OR o.order_estimated_delivery_date IS NULL
            THEN NULL

        WHEN DATEDIFF(
            o.order_delivered_customer_date,
            o.order_estimated_delivery_date
        ) > 0
            THEN 1

        ELSE 0
    END AS is_delayed

FROM orders o

JOIN customers c
    ON o.customer_id = c.customer_id

LEFT JOIN (
    SELECT
        order_id,
        SUM(price) AS item_price_total,
        SUM(freight_value) AS freight_total,
        SUM(price + freight_value) AS order_total_value
    FROM order_items
    GROUP BY order_id
) ov
    ON o.order_id = ov.order_id
"""

In [72]:
with engine.begin() as connection:
    connection.execute(text(order_features_sql))

print("order_features view created successfully.")

order_features view created successfully.


In [73]:
# CUSTOMER FEATURE
customer_features_sql = """
CREATE OR REPLACE VIEW customer_features AS
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS customer_order_count,
    ROUND(SUM(order_total_value), 2) AS customer_total_spending,
    ROUND(AVG(order_total_value), 2) AS average_order_value,
    CASE
        WHEN COUNT(DISTINCT order_id) > 1 THEN 1
        ELSE 0
    END AS repeat_customer_indicator,
    MIN(order_purchase_timestamp) AS customer_first_order_date,
    MAX(order_purchase_timestamp) AS customer_last_order_date
FROM order_features
GROUP BY customer_unique_id
"""

In [74]:
with engine.begin() as connection:
    connection.execute(text(customer_features_sql))

print("customer_features view created successfully.")

customer_features view created successfully.


In [75]:
#SELLER FEATURE
seller_features_sql = """
CREATE OR REPLACE VIEW seller_features AS
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,
    ROUND(COALESCE(SUM(oi.price), 0), 2) AS seller_revenue,
    COUNT(DISTINCT oi.order_id) AS seller_order_count,
    COUNT(oi.order_id) AS items_sold,
    ROUND(COALESCE(SUM(oi.freight_value), 0), 2) AS seller_freight_value,
    ROUND(COALESCE(AVG(oi.price), 0), 2) AS average_item_price
FROM sellers s
LEFT JOIN order_items oi
    ON s.seller_id = oi.seller_id
GROUP BY
    s.seller_id,
    s.seller_city,
    s.seller_state
"""

In [76]:
with engine.begin() as connection:
    connection.execute(text(seller_features_sql))

print("seller_features view created successfully.")

seller_features view created successfully.


In [77]:
# PRODUCT FEATURE
product_features_sql = """
CREATE OR REPLACE VIEW product_features AS
SELECT
    p.product_id,
    p.product_category_name,
    COALESCE(
        ct.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS category_english,
    COUNT(oi.order_id) AS items_sold,
    COUNT(DISTINCT oi.order_id) AS order_count,
    ROUND(COALESCE(SUM(oi.price), 0), 2) AS product_revenue,
    ROUND(COALESCE(SUM(oi.freight_value), 0), 2) AS product_freight_value
FROM products p
LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name
LEFT JOIN order_items oi
    ON p.product_id = oi.product_id
GROUP BY
    p.product_id,
    p.product_category_name,
    ct.product_category_name_english
"""

In [78]:
with engine.begin() as connection:
    connection.execute(text(product_features_sql))

print("product_features view created successfully.")

product_features view created successfully.


In [79]:
category_features_sql = """
CREATE OR REPLACE VIEW category_features AS
SELECT
    COALESCE(
        ct.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS category_name,
    COUNT(DISTINCT p.product_id) AS product_count,
    COUNT(oi.order_id) AS items_sold,
    COUNT(DISTINCT oi.order_id) AS order_count,
    ROUND(COALESCE(SUM(oi.price), 0), 2) AS category_revenue,
    ROUND(COALESCE(SUM(oi.freight_value), 0), 2) AS category_freight_value
FROM products p
LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name
LEFT JOIN order_items oi
    ON p.product_id = oi.product_id
GROUP BY
    COALESCE(
        ct.product_category_name_english,
        p.product_category_name,
        'Unknown'
    )
"""

In [80]:
with engine.begin() as connection:
    connection.execute(text(category_features_sql))

print("category_features view created successfully.")

category_features view created successfully.


In [81]:
order_enriched_features_sql = """
CREATE OR REPLACE VIEW order_enriched_features AS
SELECT
    o.*,
    c.customer_order_count,
    c.customer_total_spending,
    c.average_order_value,
    c.repeat_customer_indicator
FROM order_features o
LEFT JOIN customer_features c
    ON o.customer_unique_id = c.customer_unique_id
"""

In [82]:
with engine.begin() as connection:
    connection.execute(text(order_enriched_features_sql))

print("order_enriched_features view created successfully.")

order_enriched_features view created successfully.


In [83]:
feature_views = [
    "order_features",
    "customer_features",
    "seller_features",
    "product_features",
    "category_features",
    "order_enriched_features"
]

with engine.connect() as connection:
    for view_name in feature_views:
        result = connection.execute(
            text(f"SHOW FULL TABLES LIKE '{view_name}'")
        )
        rows = result.fetchall()

        if rows:
            print(f"✓ {view_name} exists")
        else:
            print(f"✗ {view_name} is missing")

✓ order_features exists
✓ customer_features exists
✓ seller_features exists
✓ product_features exists
✓ category_features exists
✓ order_enriched_features exists


In [84]:
order_features = pd.read_sql(
    "SELECT * FROM order_features",
    engine
)

customer_features = pd.read_sql(
    "SELECT * FROM customer_features",
    engine
)

seller_features = pd.read_sql(
    "SELECT * FROM seller_features",
    engine
)

product_features = pd.read_sql(
    "SELECT * FROM product_features",
    engine
)

category_features = pd.read_sql(
    "SELECT * FROM category_features",
    engine
)

order_enriched_features = pd.read_sql(
    "SELECT * FROM order_enriched_features",
    engine
)

print("order_features:", order_features.shape)
print("customer_features:", customer_features.shape)
print("seller_features:", seller_features.shape)
print("product_features:", product_features.shape)
print("category_features:", category_features.shape)
print("order_enriched_features:", order_enriched_features.shape)

order_features: (99441, 16)
customer_features: (96096, 7)
seller_features: (3095, 8)
product_features: (32951, 7)
category_features: (74, 6)
order_enriched_features: (99441, 20)


In [85]:
display(order_features.head())
display(customer_features.head())
display(seller_features.head())
display(product_features.head())
display(category_features.head())
display(order_enriched_features.head())

,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_price_total,freight_total,order_total_value,delivery_days,delivery_delay_days,delivery_status,is_delayed
0,5f79b5b0931d63f1a42989eb65b9da6e,00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,delivered,2017-11-14 16:08:26,2017-11-14 16:35:32,2017-11-17 15:32:08,2017-11-28 15:41:30,2017-12-04,89.80,24.94,114.74,13.0,-6.0,Early,0.0
1,a44895d095d7e0702b6a162fa2dbeced,000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,delivered,2017-07-16 09:40:32,2017-07-16 09:55:12,2017-07-19 19:09:37,2017-07-25 18:57:33,2017-08-04,54.90,12.51,67.41,9.0,-10.0,Early,0.0
2,316a104623542e4d75189bb372bc5f8d,0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,delivered,2017-02-28 11:06:43,2017-02-28 11:15:20,2017-03-01 15:24:20,2017-03-06 08:57:49,2017-03-22,179.99,15.43,195.42,5.0,-16.0,Early,0.0
3,5825ce2e88d5346438686b0bba99e5ee,0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,delivered,2017-08-16 13:09:20,2017-08-17 03:10:27,2017-08-19 11:34:29,2017-09-13 20:06:02,2017-09-14,149.90,29.45,179.35,28.0,-1.0,Early,0.0
4,0ab7fb08086d4af9141453c91878ed7a,000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,delivered,2018-04-02 13:42:17,2018-04-04 03:10:19,2018-04-04 18:11:09,2018-04-13 20:21:08,2018-04-18,93.00,14.01,107.01,11.0,-5.0,Early,0.0


,customer_unique_id,customer_order_count,customer_total_spending,average_order_value,repeat_customer_indicator,customer_first_order_date,customer_last_order_date
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,0,2018-05-10 10:56:27,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,0,2018-05-07 11:11:27,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,0,2017-03-10 21:05:03,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,0,2017-10-12 20:29:41,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,0,2017-11-14 19:45:42,2017-11-14 19:45:42


,seller_id,seller_city,seller_state,seller_revenue,seller_order_count,items_sold,seller_freight_value,average_item_price
0,0015a82c2db000af6aaaf3ae2ecb0532,santo andre,SP,2685.00,3,3,63.06,895.00
1,001cca7ae9ae17fb1caed9dfb1094831,cariacica,ES,25080.03,200,239,8854.14,104.94
2,001e6ad469a905060d959994f1b41e4f,sao goncalo,RJ,250.00,1,1,17.94,250.00
3,002100f778ceb8431b7a1020ff7ab48f,franca,SP,1234.50,51,55,793.66,22.45
4,003554e2dce176b5555353e4f3555ac8,goiania,GO,120.00,1,1,19.38,120.00


,product_id,product_category_name,category_english,items_sold,order_count,product_revenue,product_freight_value
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,perfumery,1,1,101.65,18.59
1,00088930e925c41fd95ebfe695fd2655,automotivo,auto,1,1,129.90,13.93
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,bed_bath_table,1,1,229.00,13.10
3,000b8f95fcb9e0096488278317764d19,utilidades_domesticas,housewares,2,2,117.80,39.20
4,000d9be29b5207b54e86aa1b1ac54872,relogios_presentes,watches_gifts,1,1,199.00,19.27


,category_name,product_count,items_sold,order_count,category_revenue,category_freight_value
0,agro_industry_and_commerce,74,212,182,72530.47,5843.60
1,air_conditioning,124,297,253,55024.96,6749.23
2,art,55,209,202,24202.64,4045.17
3,arts_and_craftmanship,19,24,23,1814.01,370.13
4,audio,58,364,350,50688.50,5710.44


,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_price_total,freight_total,order_total_value,delivery_days,delivery_delay_days,delivery_status,is_delayed,customer_order_count,customer_total_spending,average_order_value,repeat_customer_indicator
0,5f79b5b0931d63f1a42989eb65b9da6e,00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,delivered,2017-11-14 16:08:26,2017-11-14 16:35:32,2017-11-17 15:32:08,2017-11-28 15:41:30,2017-12-04,89.80,24.94,114.74,13.0,-6.0,Early,0.0,1,114.74,114.74,0
1,a44895d095d7e0702b6a162fa2dbeced,000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,delivered,2017-07-16 09:40:32,2017-07-16 09:55:12,2017-07-19 19:09:37,2017-07-25 18:57:33,2017-08-04,54.90,12.51,67.41,9.0,-10.0,Early,0.0,1,67.41,67.41,0
2,316a104623542e4d75189bb372bc5f8d,0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,delivered,2017-02-28 11:06:43,2017-02-28 11:15:20,2017-03-01 15:24:20,2017-03-06 08:57:49,2017-03-22,179.99,15.43,195.42,5.0,-16.0,Early,0.0,1,195.42,195.42,0
3,5825ce2e88d5346438686b0bba99e5ee,0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,delivered,2017-08-16 13:09:20,2017-08-17 03:10:27,2017-08-19 11:34:29,2017-09-13 20:06:02,2017-09-14,149.90,29.45,179.35,28.0,-1.0,Early,0.0,1,179.35,179.35,0
4,0ab7fb08086d4af9141453c91878ed7a,000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,delivered,2018-04-02 13:42:17,2018-04-04 03:10:19,2018-04-04 18:11:09,2018-04-13 20:21:08,2018-04-18,93.00,14.01,107.01,11.0,-5.0,Early,0.0,1,107.01,107.01,0


In [4]:
orders_query = """
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM orders
"""

orders_df = pd.read_sql(
    orders_query,
    con=engine
)

print("Orders:", orders_df.shape)
orders_df.head()

Orders: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17


In [5]:
order_items_query = """
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
FROM order_items
"""

order_items_df = pd.read_sql(
    order_items_query,
    con=engine
)

print("Order items:", order_items_df.shape)
order_items_df.head()

Order items: (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [6]:
# load payments
payments_query = """
SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
FROM order_payments
"""

payments_df = pd.read_sql(
    payments_query,
    con=engine
)

print("Payments:", payments_df.shape)
payments_df.head()

Payments: (103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,credit_card,2,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,credit_card,3,259.83
2,000229ec398224ef6ca0657da4fc703e,1,credit_card,5,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,credit_card,2,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,credit_card,3,218.04


In [7]:
customers_query = """
SELECT
    customer_id,
    customer_unique_id,
    customer_city,
    customer_state
FROM customers
"""

customers_df = pd.read_sql(
    customers_query,
    con=engine
)

print("Customers:", customers_df.shape)
customers_df.head()

Customers: (99441, 4)


,customer_id,customer_unique_id,customer_city,customer_state
0,00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,Osasco,SP
1,000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,Itapecerica,MG
2,0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,Nova Venecia,ES
3,0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,Mendonca,MG
4,000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,Sao Paulo,SP


In [8]:
sellers_query = """
SELECT
    seller_id,
    seller_city,
    seller_state
FROM sellers
"""

sellers_df = pd.read_sql(
    sellers_query,
    con=engine
)

print("Sellers:", sellers_df.shape)
sellers_df.head()

Sellers: (3095, 3)


,seller_id,seller_city,seller_state
0,0015a82c2db000af6aaaf3ae2ecb0532,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,cariacica,ES
2,001e6ad469a905060d959994f1b41e4f,sao goncalo,RJ
3,002100f778ceb8431b7a1020ff7ab48f,franca,SP
4,003554e2dce176b5555353e4f3555ac8,goiania,GO


In [9]:
products_query = """
SELECT
    product_id,
    product_category_name
FROM products
"""

products_df = pd.read_sql(
    products_query,
    con=engine
)

print("Products:", products_df.shape)
products_df.head()

Products: (32951, 2)


,product_id,product_category_name
0,0082684bb4a60a862baaf7a60a5845ed,None
1,00ab8a8b9fe219511dc3f178c6d79698,None
2,00d62b338366db4c4aec8547ea8f928e,None
3,0103863bf3441460142ec23c74388e4c,None
4,014fcf6bd5cd4c7ee29fb3bb618c445e,None


In [10]:
category_query = """
SELECT
    product_category_name,
    product_category_name_english
FROM category_translation
"""

category_df = pd.read_sql(
    category_query,
    con=engine
)

print("Categories:", category_df.shape)
category_df.head()

Categories: (73, 2)


,product_category_name,product_category_name_english
0,agro_industria_e_comercio,agro_industry_and_commerce
1,alimentos,food
2,alimentos_bebidas,food_drink
3,artes,art
4,artes_e_artesanato,arts_and_craftmanship


In [11]:
reviews_query = """
SELECT
    review_id,
    order_id,
    review_score
FROM order_reviews
"""

reviews_df = pd.read_sql(
    reviews_query,
    con=engine
)

print("Reviews:", reviews_df.shape)
reviews_df.head()

Reviews: (99224, 3)


,review_id,order_id,review_score
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5


In [13]:
#CELL 13
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_columns:
    orders_df[col] = pd.to_datetime(
        orders_df[col],
        errors="coerce"
    )

print(orders_df[order_date_columns].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


Total Order Value
= SUM(price + freight_value)
for each order

In [14]:
# item total
order_items_df["item_total_value"] = (
    order_items_df["price"] +
    order_items_df["freight_value"]
)

order_items_df[
    [
        "order_id",
        "price",
        "freight_value",
        "item_total_value"
    ]
].head()

,order_id,price,freight_value,item_total_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [15]:
# create order total value
order_value_df = (
    order_items_df
    .groupby("order_id", as_index=False)
    .agg(
        total_order_value=("item_total_value", "sum")
    )
)

order_value_df.head()

,order_id,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [16]:
# check total order value
print(
    "Orders with calculated total value:",
    len(order_value_df)
)

print(
    "Minimum order value:",
    order_value_df["total_order_value"].min()
)

print(
    "Maximum order value:",
    order_value_df["total_order_value"].max()
)

print(
    "Average order value:",
    order_value_df["total_order_value"].mean()
)

Orders with calculated total value: 98666
Minimum order value: 9.59
Maximum order value: 13664.08
Average order value: 160.57763809214927


Feature 2: Delivery Days
Delivery Days
=
Delivered Customer Date
-
Order Purchase Date

In [17]:
# Calculate Delivery Days
orders_df["delivery_days"] = (
    orders_df["order_delivered_customer_date"]
    - orders_df["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

orders_df["delivery_days"] = orders_df["delivery_days"].round(2)

In [18]:
# inspect delivery days
orders_df[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
].head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-13 08:59:02,2017-09-20 23:43:48,7.61
1,00018f77f2f0320c557190d7a144bdd3,2017-04-26 10:53:06,2017-05-12 16:04:24,16.22
2,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,2018-01-22 13:19:16,7.95
3,00024acbcdf0a6daa1e931b038114c75,2018-08-08 10:00:35,2018-08-14 13:32:39,6.15
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-04 13:57:51,2017-03-01 16:42:31,25.11
5,00048cc3ae777c65dbb7d2a0634bc1ea,2017-05-15 21:42:34,2017-05-22 13:44:35,6.67
6,00054e8431b9d7675808bcb819fb4a32,2017-12-10 11:53:48,2017-12-18 22:03:38,8.42
7,000576fe39319847cbb9d288c5617fa6,2018-07-04 12:08:27,2018-07-09 14:04:07,5.08
8,0005a1a1728c9d785b8e2b08b904576c,2018-03-19 18:40:33,2018-03-29 18:17:31,9.98
9,0005f50442cb953dcd1d21e1fb923495,2018-07-02 13:59:39,2018-07-04 17:28:31,2.15


In [19]:
# validate delivery days
print(
    "Valid delivery days:",
    orders_df["delivery_days"].notna().sum()
)

print(
    "Missing delivery days:",
    orders_df["delivery_days"].isna().sum()
)

print(
    "Minimum delivery days:",
    orders_df["delivery_days"].min()
)

print(
    "Maximum delivery days:",
    orders_df["delivery_days"].max()
)

print(
    "Average delivery days:",
    orders_df["delivery_days"].mean()
)

Valid delivery days: 96453
Missing delivery days: 2988
Minimum delivery days: 0.53
Maximum delivery days: 209.63
Average delivery days: 12.560049765170598


Feature 3: Delivery Delay
Actual Delivery Date
        vs
Estimated Delivery Date

Delivery Delay
=
Actual Delivery Date
-
Estimated Delivery Date

In [20]:
# calculate delivery delay
orders_df["delivery_delay_days"] = (
    orders_df["order_delivered_customer_date"]
    - orders_df["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

orders_df["delivery_delay_days"] = (
    orders_df["delivery_delay_days"].round(2)
)

In [21]:
# inspect delivery delay
orders_df[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay_days"
    ]
].head(10)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-20 23:43:48,2017-09-29,-8.01
1,00018f77f2f0320c557190d7a144bdd3,2017-05-12 16:04:24,2017-05-15,-2.33
2,000229ec398224ef6ca0657da4fc703e,2018-01-22 13:19:16,2018-02-05,-13.44
3,00024acbcdf0a6daa1e931b038114c75,2018-08-14 13:32:39,2018-08-20,-5.44
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-03-01 16:42:31,2017-03-17,-15.30
5,00048cc3ae777c65dbb7d2a0634bc1ea,2017-05-22 13:44:35,2017-06-06,-14.43
6,00054e8431b9d7675808bcb819fb4a32,2017-12-18 22:03:38,2018-01-04,-16.08
7,000576fe39319847cbb9d288c5617fa6,2018-07-09 14:04:07,2018-07-25,-15.41
8,0005a1a1728c9d785b8e2b08b904576c,2018-03-29 18:17:31,2018-03-29,0.76
9,0005f50442cb953dcd1d21e1fb923495,2018-07-04 17:28:31,2018-07-23,-18.27


In [22]:
# create delivery status
orders_df["delivery_status"] = np.select(
    [
        orders_df["delivery_delay_days"].isna(),
        orders_df["delivery_delay_days"] > 0,
        orders_df["delivery_delay_days"] <= 0
    ],
    [
        "Unknown",
        "Delayed",
        "On Time"
    ],
    default="Unknown"
)

In [23]:
# check delivery status
orders_df["delivery_status"].value_counts(dropna=False)

On Time    88632
Delayed     7821
Unknown     2988
Name: delivery_status, dtype: int64

Feature 4: Customer Order Count


In [24]:
# connect orders with customers
customer_orders_df = orders_df.merge(
    customers_df[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

print(customer_orders_df.shape)

customer_orders_df.head()

(99441, 14)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status,customer_unique_id,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.61,-8.01,On Time,871766c5855e863f6eccc05f988b23cb,Campos Dos Goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.22,-2.33,On Time,eb28e67c4c0b83846050ddfb8a35d051,Santa Fe Do Sul,SP
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.95,-13.44,On Time,3818d81c6709e39d06b2738a8d3a2474,Para De Minas,MG
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.15,-5.44,On Time,af861d436cfc08b2c2ddefd0ba074622,Atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.11,-15.30,On Time,64b576fb70d441e8f1b2d7d446e483c5,Varzea Paulista,SP


In [25]:
# customer order count
customer_order_count_df = (
    customer_orders_df
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_order_count=("order_id", "nunique")
    )
)

customer_order_count_df.head()

,customer_unique_id,customer_order_count
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1


Feature 5: Customer Total Spending

In [26]:
# add order value to customer data
customer_orders_df = customer_orders_df.merge(
    order_value_df,
    on="order_id",
    how="left"
)

customer_orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status,customer_unique_id,customer_city,customer_state,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.61,-8.01,On Time,871766c5855e863f6eccc05f988b23cb,Campos Dos Goytacazes,RJ,72.19
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.22,-2.33,On Time,eb28e67c4c0b83846050ddfb8a35d051,Santa Fe Do Sul,SP,259.83
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.95,-13.44,On Time,3818d81c6709e39d06b2738a8d3a2474,Para De Minas,MG,216.87
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.15,-5.44,On Time,af861d436cfc08b2c2ddefd0ba074622,Atibaia,SP,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.11,-15.30,On Time,64b576fb70d441e8f1b2d7d446e483c5,Varzea Paulista,SP,218.04


In [27]:
# customer total spending
customer_spending_df = (
    customer_orders_df
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_total_spending=("total_order_value", "sum")
    )
)

customer_spending_df.head()

,customer_unique_id,customer_total_spending
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19
2,0000f46a3911fa3c0805444483337064,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89


In [28]:
# customer spending validation
print(
    "Customers:",
    customer_spending_df["customer_unique_id"].nunique()
)

print(
    "Minimum spending:",
    customer_spending_df["customer_total_spending"].min()
)

print(
    "Maximum spending:",
    customer_spending_df["customer_total_spending"].max()
)

print(
    "Average spending:",
    customer_spending_df["customer_total_spending"].mean()
)

Customers: 96096
Minimum spending: 0.0
Maximum spending: 13664.08
Average spending: 164.87214077589078


Feature 6: Average Order Value

In [29]:
customer_features_df = customer_order_count_df.merge(
    customer_spending_df,
    on="customer_unique_id",
    how="outer"
)

customer_features_df.head()

,customer_unique_id,customer_order_count,customer_total_spending
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19
2,0000f46a3911fa3c0805444483337064,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,196.89


In [30]:
# calculate average order value
customer_features_df["average_order_value"] = (
    customer_features_df["customer_total_spending"]
    / customer_features_df["customer_order_count"]
)

customer_features_df["average_order_value"] = (
    customer_features_df["average_order_value"].round(2)
)

In [31]:
# validate AOV
customer_features_df[
    [
        "customer_unique_id",
        "customer_order_count",
        "customer_total_spending",
        "average_order_value"
    ]
].head(10)

,customer_unique_id,customer_order_count,customer_total_spending,average_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89
5,0004bd2a26a76fe21f786e4fbd80607f,1,166.98,166.98
6,00050ab1314c0e55a6ca13cf7181fecf,1,35.38,35.38
7,00053a61a98854899e70ed204dd4bafe,1,419.18,419.18
8,0005e1862207bf6ccc02e4228effd9a0,1,150.12,150.12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,1,129.76,129.76


Feature 7: Repeat Customer Indicator

In [32]:
customer_features_df["repeat_customer"] = np.where(
    customer_features_df["customer_order_count"] > 1,
    1,
    0
)

customer_features_df.head()

,customer_unique_id,customer_order_count,customer_total_spending,average_order_value,repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,0
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,0


In [33]:
customer_features_df["repeat_customer"].value_counts()

0    93099
1     2997
Name: repeat_customer, dtype: int64

In [34]:
repeat_customer_percentage = (
    customer_features_df["repeat_customer"].mean() * 100
)

print(
    f"Repeat customer percentage: "
    f"{repeat_customer_percentage:.2f}%"
)

Repeat customer percentage: 3.12%


Feature 8: Seller Revenue

Seller Revenue
=
SUM(price + freight_value)

In [35]:
seller_revenue_df = (
    order_items_df
    .groupby("seller_id", as_index=False)
    .agg(
        seller_revenue=("item_total_value", "sum")
    )
)

seller_revenue_df.head()

,seller_id,seller_revenue
0,0015a82c2db000af6aaaf3ae2ecb0532,2748.06
1,001cca7ae9ae17fb1caed9dfb1094831,33934.17
2,001e6ad469a905060d959994f1b41e4f,267.94
3,002100f778ceb8431b7a1020ff7ab48f,2028.16
4,003554e2dce176b5555353e4f3555ac8,139.38


In [36]:
seller_revenue_df["seller_revenue"] = (
    seller_revenue_df["seller_revenue"].round(2)
)

In [37]:
print(
    "Number of sellers with sales:",
    seller_revenue_df["seller_id"].nunique()
)

print(
    "Minimum seller revenue:",
    seller_revenue_df["seller_revenue"].min()
)

print(
    "Maximum seller revenue:",
    seller_revenue_df["seller_revenue"].max()
)

print(
    "Average seller revenue:",
    seller_revenue_df["seller_revenue"].mean()
)

Number of sellers with sales: 3095
Minimum seller revenue: 12.22
Maximum seller revenue: 249640.7
Average seller revenue: 5119.080206785137


Feature 9: Seller Order Count

In [38]:
seller_order_count_df = (
    order_items_df
    .groupby("seller_id", as_index=False)
    .agg(
        seller_order_count=("order_id", "nunique")
    )
)

seller_order_count_df.head()

,seller_id,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,3
1,001cca7ae9ae17fb1caed9dfb1094831,200
2,001e6ad469a905060d959994f1b41e4f,1
3,002100f778ceb8431b7a1020ff7ab48f,51
4,003554e2dce176b5555353e4f3555ac8,1


In [39]:
seller_features_df = seller_revenue_df.merge(
    seller_order_count_df,
    on="seller_id",
    how="outer"
)

seller_features_df.head()

,seller_id,seller_revenue,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,2748.06,3
1,001cca7ae9ae17fb1caed9dfb1094831,33934.17,200
2,001e6ad469a905060d959994f1b41e4f,267.94,1
3,002100f778ceb8431b7a1020ff7ab48f,2028.16,51
4,003554e2dce176b5555353e4f3555ac8,139.38,1


In [40]:
# Add seller Location
seller_features_df = seller_features_df.merge(
    sellers_df,
    on="seller_id",
    how="left"
)

seller_features_df.head()

,seller_id,seller_revenue,seller_order_count,seller_city,seller_state
0,0015a82c2db000af6aaaf3ae2ecb0532,2748.06,3,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,33934.17,200,cariacica,ES
2,001e6ad469a905060d959994f1b41e4f,267.94,1,sao goncalo,RJ
3,002100f778ceb8431b7a1020ff7ab48f,2028.16,51,franca,SP
4,003554e2dce176b5555353e4f3555ac8,139.38,1,goiania,GO


In [41]:
# Add product category
product_category_df = products_df.merge(
    category_df,
    on="product_category_name",
    how="left"
)

product_category_df.head()

,product_id,product_category_name,product_category_name_english
0,0082684bb4a60a862baaf7a60a5845ed,None,NaN
1,00ab8a8b9fe219511dc3f178c6d79698,None,NaN
2,00d62b338366db4c4aec8547ea8f928e,None,NaN
3,0103863bf3441460142ec23c74388e4c,None,NaN
4,014fcf6bd5cd4c7ee29fb3bb618c445e,None,NaN


In [42]:
# Add category to order items
order_items_enriched_df = order_items_df.merge(
    product_category_df[
        [
            "product_id",
            "product_category_name",
            "product_category_name_english"
        ]
    ],
    on="product_id",
    how="left"
)

order_items_enriched_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_total_value,product_category_name,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,cool_stuff,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,pet_shop,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,moveis_decoracao,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,perfumaria,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,ferramentas_jardim,garden_tools


In [43]:
# Add order value
order_features_df = orders_df.merge(
    order_value_df,
    on="order_id",
    how="left"
)

order_features_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.61,-8.01,On Time,72.19
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.22,-2.33,On Time,259.83
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.95,-13.44,On Time,216.87
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.15,-5.44,On Time,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.11,-15.30,On Time,218.04


In [44]:
# Add Customer Information
order_features_df = order_features_df.merge(
    customers_df[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

order_features_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status,total_order_value,customer_unique_id,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.61,-8.01,On Time,72.19,871766c5855e863f6eccc05f988b23cb,Campos Dos Goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.22,-2.33,On Time,259.83,eb28e67c4c0b83846050ddfb8a35d051,Santa Fe Do Sul,SP
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.95,-13.44,On Time,216.87,3818d81c6709e39d06b2738a8d3a2474,Para De Minas,MG
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.15,-5.44,On Time,25.78,af861d436cfc08b2c2ddefd0ba074622,Atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.11,-15.30,On Time,218.04,64b576fb70d441e8f1b2d7d446e483c5,Varzea Paulista,SP


In [45]:
# Add review score
order_review_score_df = (
    reviews_df
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean")
    )
)

order_review_score_df.head()

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [46]:
# Merge Review Score
order_features_df = order_features_df.merge(
    order_review_score_df,
    on="order_id",
    how="left"
)

order_features_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status,total_order_value,customer_unique_id,customer_city,customer_state,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.61,-8.01,On Time,72.19,871766c5855e863f6eccc05f988b23cb,Campos Dos Goytacazes,RJ,5.0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.22,-2.33,On Time,259.83,eb28e67c4c0b83846050ddfb8a35d051,Santa Fe Do Sul,SP,4.0
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.95,-13.44,On Time,216.87,3818d81c6709e39d06b2738a8d3a2474,Para De Minas,MG,5.0
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.15,-5.44,On Time,25.78,af861d436cfc08b2c2ddefd0ba074622,Atibaia,SP,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.11,-15.30,On Time,218.04,64b576fb70d441e8f1b2d7d446e483c5,Varzea Paulista,SP,5.0


Add Customer Features to Order Dataset

In [47]:
order_features_df = order_features_df.merge(
    customer_features_df[
        [
            "customer_unique_id",
            "customer_order_count",
            "customer_total_spending",
            "average_order_value",
            "repeat_customer"
        ]
    ],
    on="customer_unique_id",
    how="left"
)

order_features_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status,total_order_value,customer_unique_id,customer_city,customer_state,review_score,customer_order_count,customer_total_spending,average_order_value,repeat_customer
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.61,-8.01,On Time,72.19,871766c5855e863f6eccc05f988b23cb,Campos Dos Goytacazes,RJ,5.0,1,72.19,72.19,0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.22,-2.33,On Time,259.83,eb28e67c4c0b83846050ddfb8a35d051,Santa Fe Do Sul,SP,4.0,2,284.56,142.28,1
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.95,-13.44,On Time,216.87,3818d81c6709e39d06b2738a8d3a2474,Para De Minas,MG,5.0,1,216.87,216.87,0
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.15,-5.44,On Time,25.78,af861d436cfc08b2c2ddefd0ba074622,Atibaia,SP,4.0,1,25.78,25.78,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.11,-15.30,On Time,218.04,64b576fb70d441e8f1b2d7d446e483c5,Varzea Paulista,SP,5.0,1,218.04,218.04,0


Validate the Main Feature Dataset

In [48]:
print("Order feature dataset shape:", order_features_df.shape)

Order feature dataset shape: (99441, 20)


In [49]:
print("Columns:")
for col in order_features_df.columns:
    print(col)

Columns:
order_id
customer_id
order_status
order_purchase_timestamp
order_approved_at
order_delivered_carrier_date
order_delivered_customer_date
order_estimated_delivery_date
delivery_days
delivery_delay_days
delivery_status
total_order_value
customer_unique_id
customer_city
customer_state
review_score
customer_order_count
customer_total_spending
average_order_value
repeat_customer


In [50]:
# check missing values
order_features_df[
    [
        "total_order_value",
        "delivery_days",
        "delivery_delay_days",
        "delivery_status",
        "customer_order_count",
        "customer_total_spending",
        "average_order_value",
        "repeat_customer",
        "review_score"
    ]
].isna().sum()

total_order_value           775
delivery_days              2988
delivery_delay_days        2988
delivery_status               0
customer_order_count          0
customer_total_spending       0
average_order_value           0
repeat_customer               0
review_score                768
dtype: int64

In [51]:
# check duplicate order
print(
    "Duplicate order IDs:",
    order_features_df["order_id"].duplicated().sum()
)

Duplicate order IDs: 0


In [52]:
#check feature statistics
order_features_df[
    [
        "total_order_value",
        "delivery_days",
        "delivery_delay_days",
        "customer_order_count",
        "customer_total_spending",
        "average_order_value",
        "review_score"
    ]
].describe()

,total_order_value,delivery_days,delivery_delay_days,customer_order_count,customer_total_spending,average_order_value,review_score
count,98666.000000,96453.000000,96453.000000,99441.000000,99441.000000,99441.000000,98673.000000
mean,160.577638,12.560050,-11.176113,1.079223,170.650658,159.326168,4.086793
std,220.466087,9.546957,10.183729,0.396154,235.775976,218.578974,1.346274
min,9.590000,0.530000,-146.020000,1.000000,0.000000,0.000000,1.000000
25%,61.980000,6.770000,-16.240000,1.000000,63.370000,61.850000,4.000000
50%,105.290000,10.220000,-11.940000,1.000000,110.200000,105.280000,5.000000
75%,176.870000,15.720000,-6.390000,1.000000,188.650000,175.990000,5.000000
max,13664.080000,209.630000,188.980000,17.000000,13664.080000,13664.080000,5.000000


In [53]:
# validate seller feature
print("Seller feature dataset shape:", seller_features_df.shape)

seller_features_df.head()

Seller feature dataset shape: (3095, 5)


,seller_id,seller_revenue,seller_order_count,seller_city,seller_state
0,0015a82c2db000af6aaaf3ae2ecb0532,2748.06,3,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,33934.17,200,cariacica,ES
2,001e6ad469a905060d959994f1b41e4f,267.94,1,sao goncalo,RJ
3,002100f778ceb8431b7a1020ff7ab48f,2028.16,51,franca,SP
4,003554e2dce176b5555353e4f3555ac8,139.38,1,goiania,GO


In [54]:
print(
    "Duplicate seller IDs:",
    seller_features_df["seller_id"].duplicated().sum()
)

Duplicate seller IDs: 0


Validate Customer Feature

In [55]:
print("Customer feature dataset shape:", customer_features_df.shape)

customer_features_df.head()

Customer feature dataset shape: (96096, 5)


,customer_unique_id,customer_order_count,customer_total_spending,average_order_value,repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,0
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,0


In [56]:
print(
    "Duplicate customer IDs:",
    customer_features_df["customer_unique_id"].duplicated().sum()
)

Duplicate customer IDs: 0


Create Category-Level Features

In [57]:
category_sales_df = (
    order_items_enriched_df
    .groupby(
        [
            "product_category_name",
            "product_category_name_english"
        ],
        dropna=False
    )
    .agg(
        category_revenue=("item_total_value", "sum"),
        category_item_count=("order_item_id", "count"),
        category_order_count=("order_id", "nunique")
    )
    .reset_index()
)

category_sales_df.head()

,product_category_name,product_category_name_english,category_revenue,category_item_count,category_order_count
0,agro_industria_e_comercio,agro_industry_and_commerce,78374.07,212,182
1,alimentos,food,36664.44,510,450
2,alimentos_bebidas,food_drink,19687.47,278,227
3,artes,art,28247.81,209,202
4,artes_e_artesanato,arts_and_craftmanship,2184.14,24,23


Create Final Feature Summary

In [58]:
print("========== FEATURE ENGINEERING SUMMARY ==========")

print(
    "Total Order Value: created"
)

print(
    "Delivery Days: created"
)

print(
    "Delivery Delay: created"
)

print(
    "Customer Order Count: created"
)

print(
    "Customer Total Spending: created"
)

print(
    "Average Order Value: created"
)

print(
    "Seller Revenue: created"
)

print(
    "Seller Order Count: created"
)

print(
    "Repeat Customer Indicator: created"
)

========== FEATURE ENGINEERING SUMMARY ==========
Total Order Value: created
Delivery Days: created
Delivery Delay: created
Customer Order Count: created
Customer Total Spending: created
Average Order Value: created
Seller Revenue: created
Seller Order Count: created
Repeat Customer Indicator: created


In [59]:
features_path = "../data/features/"

os.makedirs(features_path, exist_ok=True)

print("Feature folder ready:", features_path)

Feature folder ready: ../data/features/


In [60]:
order_features_df.to_csv(
    os.path.join(
        features_path,
        "order_features.csv"
    ),
    index=False
)

print("Order features saved.")

Order features saved.


In [61]:
customer_features_df.to_csv(
    os.path.join(
        features_path,
        "customer_features.csv"
    ),
    index=False
)

print("Customer features saved.")

Customer features saved.


In [62]:
seller_features_df.to_csv(
    os.path.join(
        features_path,
        "seller_features.csv"
    ),
    index=False
)

print("Seller features saved.")

Seller features saved.


In [63]:
category_sales_df.to_csv(
    os.path.join(
        features_path,
        "category_sales_features.csv"
    ),
    index=False
)

print("Category features saved.")

Category features saved.


In [64]:
print("Saved feature files:")

for filename in os.listdir(features_path):
    print(filename)

Saved feature files:
category_sales_features.csv
customer_features.csv
order_features.csv
seller_features.csv


In [65]:
saved_order_features = pd.read_csv(
    os.path.join(
        features_path,
        "order_features.csv"
    )
)

print(
    "Saved order feature shape:",
    saved_order_features.shape
)

saved_order_features.head()

Saved order feature shape: (99441, 20)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status,total_order_value,customer_unique_id,customer_city,customer_state,review_score,customer_order_count,customer_total_spending,average_order_value,repeat_customer
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,7.61,-8.01,On Time,72.19,871766c5855e863f6eccc05f988b23cb,Campos Dos Goytacazes,RJ,5.0,1,72.19,72.19,0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,16.22,-2.33,On Time,259.83,eb28e67c4c0b83846050ddfb8a35d051,Santa Fe Do Sul,SP,4.0,2,284.56,142.28,1
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,7.95,-13.44,On Time,216.87,3818d81c6709e39d06b2738a8d3a2474,Para De Minas,MG,5.0,1,216.87,216.87,0
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,6.15,-5.44,On Time,25.78,af861d436cfc08b2c2ddefd0ba074622,Atibaia,SP,4.0,1,25.78,25.78,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,25.11,-15.30,On Time,218.04,64b576fb70d441e8f1b2d7d446e483c5,Varzea Paulista,SP,5.0,1,218.04,218.04,0


In [66]:
required_features = [
    "total_order_value",
    "delivery_days",
    "delivery_delay_days",
    "delivery_status",
    "customer_order_count",
    "customer_total_spending",
    "average_order_value",
    "repeat_customer"
]

missing_features = [
    feature
    for feature in required_features
    if feature not in saved_order_features.columns
]

if len(missing_features) == 0:
    print("All required order-level features are present.")
else:
    print("Missing features:", missing_features)

All required order-level features are present.


In [67]:
print("================================================")
print("STEP 7 FEATURE ENGINEERING COMPLETED")
print("================================================")

print("Created:")
print("1. Total Order Value")
print("2. Delivery Days")
print("3. Delivery Delay")
print("4. Customer Order Count")
print("5. Customer Total Spending")
print("6. Average Order Value")
print("7. Seller Revenue")
print("8. Seller Order Count")
print("9. Repeat Customer Indicator")

print("\nFeature datasets saved successfully.")
print("================================================")

STEP 7 FEATURE ENGINEERING COMPLETED
Created:
1. Total Order Value
2. Delivery Days
3. Delivery Delay
4. Customer Order Count
5. Customer Total Spending
6. Average Order Value
7. Seller Revenue
8. Seller Order Count
9. Repeat Customer Indicator

Feature datasets saved successfully.
